### Notebook to process and manipulate coverage geopackage
- Works for gpkg that have been created with Create_cov.sh
- Iterates over user defined directory path and processes all geopackages inside whose filenames end with "_Area.gpkg"
- *TODO:* Define and add metadata; add logic to append metadata
- *TODO:* Add Osis link -> access API to match Osis link to correct cruise

In [2]:
import shutil
import geopandas as gpd
import pandas as pd
import os
import numpy as np
from pathlib import Path
import glob

#### 3. Process and add metadata to coverage polygon
- remove zero value box around track
- dissolve fields if multiple are present
- calculate area of swath coverage
- add name of original raster
- **⚡ Change path name to the folder where the geopackage is located in**

In [4]:
gpkg_path = "/path/to/gpkg" # folder that contains coverage geopackages that shall be processed
gpkg_path = "/Users/mschumacher/Docs_Data/Bathy/Processing/"

In [5]:
for gpkg in glob.glob(f"{gpkg_path}/*_Area.gpkg"):
    gpkg_df = gpd.read_file(gpkg, index_col = False)
    gpkg_df_red = gpkg_df.drop(gpkg_df[gpkg_df['DN'] == 0].index)
    gpkg_diss = gpkg_df_red.dissolve(by = 'DN')
    base_grid_filename = os.path.basename(gpkg).replace("_Area", "")
    gpkg_diss['Filename'] = base_grid_filename
    gpkg_diss['Area [km2]'] = np.sum(gpkg_diss['geometry'].area)/(1000*1000)
    out_gpkg = gpkg.replace("Area", "Coverage")
    print(out_gpkg)
    gpkg_diss.to_file(out_gpkg, driver='GPKG', mode='w')

/Users/mschumacher/Docs_Data/Bathy/Processing/StarfishLog_20240822_132111_7mm_ch1_EPSG32632_Coverage.gpkg


/Users/mschumacher/Docs_Data/VS/miniconda3/envs/.gis/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: driver GPKG does not support open option INDEX_COL
  return ogr_read(
